# Token Cost vs. Performance Tradeoff

### Source : PROMPT_AND_INSTRUCTION_DESIGN_2\token_tradeoff_5.ipynb


### Objective:
Given experiment tracking results comparing two system prompt
configurations on a Databricks agent, determine whether the higher-token configuration
is justified and identify the specific prompt element driving the cost-performance tradeoff.

## Setup

In [ ]:
import os, json, uuid, re
import sys
# Add parent directory to path to access shared utilities
sys.path.append(os.path.abspath(".."))
from _utils.sql_utils import sql_call   # shared SQL helper
from dotenv import load_dotenv
load_dotenv()   # reads DATABRICKS_HOST, TOKEN, SQL_WAREHOUSE_ID, MLFLOW_TRACKING_URI
assert os.getenv("DATABRICKS_HOST"),     "DATABRICKS_HOST not set"
assert os.getenv("DATABRICKS_TOKEN"),    "DATABRICKS_TOKEN not set"
assert os.getenv("SQL_WAREHOUSE_ID"),    "SQL_WAREHOUSE_ID not set"
assert os.getenv("MLFLOW_TRACKING_URI"), "MLFLOW_TRACKING_URI not set"

## Imports and config

In [ ]:
import mlflow, tiktoken
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks
from langchain_core.messages import HumanMessage, SystemMessage

# Unity Catalog coordinates -- prompts registered as CATALOG.SCHEMA.name
CATALOG, SCHEMA = "demo", "context"
SUFFIX          = uuid.uuid4().hex[:6]   # unique suffix per run
MODEL_ENDPOINT  = "databricks-meta-llama-3-3-70b-instruct"

# Workspace client -- credentials loaded from .env
w = WorkspaceClient()

# MLflow experiment -- links to UC prompt registry via tag
mlflow.set_experiment("/Users/oliver@mlops-media.com/chapter_2_token_tradeoff")
mlflow.set_experiment_tags({"mlflow.promptRegistryLocation": f"{CATALOG}.{SCHEMA}"})
print(f"host={w.config.host}  endpoint={MODEL_ENDPOINT}  suffix={SUFFIX}")

In [ ]:
# Invoke the serving endpoint via LangChain ChatDatabricks wrapper
llm = ChatDatabricks(endpoint=MODEL_ENDPOINT)
def call_llm(system, user, max_tokens=512):
    msgs = ([SystemMessage(content=system)] if system else []) + [HumanMessage(content=user)]
    return llm.invoke(msgs, max_tokens=max_tokens).content

# tiktoken (cl100k_base) as a cost-proxy token counter for prompt budgeting
# Note: Llama tokenizer differs slightly -- this is a conservative estimate
enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text):
    return len(enc.encode(text or ""))

## 1 - Register 4 prompt versions in Unity Catalog

In [ ]:
# 4 versions isolate which element drives the tradeoff:
#   v1: baseline -- no framing, no constraint
#   v2: + role framing only
#   v3: + length constraint only (2 sentences)
#   v4: full config (role + constraint + guidelines + output label)
# Docs: mlflow.genai.register_prompt requires mlflow[databricks]>=3.1.0
#       and CREATE FUNCTION + EXECUTE + MANAGE on the UC schema
# Ref: https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry

PROMPT_NAME = f"{CATALOG}.{SCHEMA}.tradeoff_{SUFFIX}"

VERSIONS = [
    ("Summarize this text: {{content}}", "v1: baseline"),
    ("You are an expert summarizer.\nSummarize this text: {{content}}", "v2: +role only"),
    ("Summarize in *exactly* 2 sentences: {{content}}", "v3: +constraint only"),
    ("You are an expert summarizer. Create a summary in *exactly* 2 sentences.\n\n"
     "Guidelines:\n- Include ALL core facts\n- Use clear, concise language\n"
     "- Maintain factual accuracy\n- Write for a general audience\n\n"
     "Content: {{content}}\n\nSummary:", "v4: full config"),
]
prompts = []
for tmpl, msg in VERSIONS:
    p = mlflow.genai.register_prompt(name=PROMPT_NAME, template=tmpl, commit_message=msg)
    prompts.append(p)
    print(f"  v{p.version}: {count_tokens(tmpl):>4} tokens -- {msg}")

## 2 - Eval dataset and scorers

In [ ]:
! pip install databricks-agents

In [ ]:
# mlflow.genai datasets + Correctness scorer
# Correctness uses expected_facts (flexible match, no word-for-word required)
from mlflow.genai.datasets import create_dataset, get_dataset
from mlflow.genai.scorers import Correctness
from mlflow.genai.judges import make_judge
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

DS_NAME = f"{CATALOG}.{SCHEMA}.tradeoff_eval_{SUFFIX}"
try:    ds = get_dataset(name=DS_NAME)
except: ds = create_dataset(name=DS_NAME)

# 4 diverse examples to make compliance differences visible
ds = ds.merge_records([
    {"inputs": {"content": "Remote work changed collaboration. Companies adopted digital tools. "
                           "Productivity held but culture and work-life balance are strained. "
                           "Global hiring accelerated."},
     "expectations": {"expected_facts": ["remote work", "digital tools", "productivity",
                                         "culture", "global hiring"]}},
    {"inputs": {"content": "EVs gain adoption as batteries improve. Government incentives help. "
                           "Challenges: upfront cost, rural charging gaps, battery life. "
                           "Market expected to grow next decade."},
     "expectations": {"expected_facts": ["EV adoption", "battery", "incentives",
                                         "upfront cost", "rural charging", "growth"]}},
    {"inputs": {"content": "AI transforms healthcare: imaging, drug discovery, personalised care. "
                           "ML detects diseases earlier. Concerns: privacy, bias, regulation."},
     "expectations": {"expected_facts": ["AI healthcare", "imaging", "drug discovery",
                                         "privacy", "regulation"]}},
    {"inputs": {"content": "Climate tech investment hit a record in 2023 driven by IRA subsidies. "
                           "Solar and storage led. Critics note slow grid permitting as a bottleneck."},
     "expectations": {"expected_facts": ["climate tech", "record investment", "IRA", "solar",
                                         "storage", "grid permitting"]}}
])

In [ ]:
# paragraph_compliance judge: checks single-paragraph prose format (no bullets)
# make_judge wraps an LLM call -- model defaults to Databricks-hosted judge LLM
paragraph_compliance=make_judge("paragraph_compliance",
    instructions=(
        "Is this a single prose paragraph with no bullet points or numbered lists?\n"
        "Summary: {{ outputs }}\n"
        "Return true for a single paragraph, false otherwise."
    ),
    feedback_value_type=bool,
)
# Correctness() uses expected_facts for flexible fact-coverage evaluation
scorers = [Correctness(), paragraph_compliance]
print(f"Dataset: {DS_NAME} | scorers: {[type(s).__name__ for s in scorers]}")

## 3 - Evaluate all 4 versions with prompt_tokens logged

In [ ]:
# make_fn: loads the registered prompt from UC, calls the serving endpoint
# @mlflow.trace decorates summarize() to capture inputs/outputs in MLflow
def make_fn(pname, ver):
    @mlflow.trace
    def summarize(content):
        p = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{pname}/{ver}")
        r = w.serving_endpoints.query(
            name=MODEL_ENDPOINT,
            messages=[ChatMessage(role=ChatMessageRole.USER,
                                  content=p.format(content=content))],
            temperature=0.1,
        )
        return {"summary": r.choices[0].message.content}
    return summarize

In [ ]:
# Key addition vs prompt_registry.ipynb: log prompt_tokens as a metric
# score_per_100_tok reveals whether the quality gain justifies the extra cost
results_D = {}
for ver in range(1, 5):
    pobj  = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{PROMPT_NAME}/{ver}")
    ptoks = count_tokens(pobj.template)
    print(f"\nEvaluating v{ver} ({ptoks} tokens)...")
    with mlflow.start_run(run_name=f"D_v{ver}_{SUFFIX}"):
        mlflow.log_param("version", ver)
        mlflow.log_metric("prompt_tokens", ptoks)   # key addition
        ev = mlflow.genai.evaluate(
            predict_fn=make_fn(PROMPT_NAME, ver), data=ds, scorers=scorers)
        corr = ev.metrics.get("correctness/mean", 0)
        comp = ev.metrics.get("paragraph_compliance/mean", 0)
        cmp  = round(0.7*corr + 0.3*comp, 3)
        s100 = round(cmp/ptoks*100, 5)
        mlflow.log_metric("correctness", corr)
        mlflow.log_metric("compliance",  comp)
        mlflow.log_metric("composite",   cmp)
        mlflow.log_metric("score_per_100_tok", s100)
    results_D[f"v{ver}"] = {"tokens":ptoks,"corr":corr,"comp":comp,"cmp":cmp,"s100":s100}
    print(f"  corr={corr:.2f}  comp={comp:.2f}  composite={cmp:.3f}  s/100tok={s100:.5f}")

## 4 -- Comparison table and LLM verdict

In [ ]:
# Print comparison table: lets the viewer see which version hits the Pareto frontier
print(f"\n{'Ver':<5} {'Tokens':>7} {'Corr':>6} {'Comp':>6} {'Composite':>10} {'S/100tok':>10}")
print("-"*52)
for v, r in results_D.items():
    print(f"{v:<5} {r['tokens']:>7} {r['corr']:>6.2f} {r['comp']:>6.2f}"
          f" {r['cmp']:>10.3f} {r['s100']:>10.5f}")

# LLM verdict: answers both exam questions from the measured results
vp = (
    "Evaluate 4 prompt versions for a Databricks summarisation agent.\n"
    f"Results: {json.dumps(results_D, indent=2)}\n\n"
    "v1=baseline  v2=+role  v3=+length-constraint  v4=role+constraint+guidelines+label\n\n"
    "Answer in 4-6 sentences:\n"
    "Q1: Is v4 (highest token) justified vs v3? Cite metrics.\n"
    "Q2: Which single element most drives compliance gain? "
    "Which could be cut from v4 while keeping most of the gain?"
)
verdict = call_llm("", vp, 400)
print(f"\n=== Exam verdict ===\n{verdict}")
with mlflow.start_run(run_name=f"D_verdict_{SUFFIX}"):
    mlflow.log_text(verdict, "tradeoff_verdict.txt")
    [mlflow.log_metric(f"{v}_{k}", val)
     for v, r in results_D.items() for k, val in r.items()]